# Ejecución de inferencia y harness con componente visual (M4)

Este notebook prueba el flujo **imagen → OCR/VLM → NER → harness**, es decir, los
cambios que permiten que las entidades no vengan del texto del gold set sino de
**transcribir imágenes** de notas clínicas con el módulo `M4/vision`.

Qué cambia respecto a `start_inference_harness.ipynb`:

- `M2/run_inference/run_inference.py` ahora tiene un bloque `fuente:` en el config.
  Con `fuente.tipo: vision` transcribe las imágenes del manifiesto de M4 y usa
  ese texto como entrada del NER. El NER, el chunking y el formato del cache son
  los mismos; solo cambia de dónde sale el texto.
- Se corre con `M2/run_inference/config_vision.yaml` (produce
  `M2/predictions/vision_<backend>.json` + un sidecar `*.vision_meta.json`).
- El harness se corre con `M2/harness/config_vision.yaml` (escribe en
  `M2/outputs_vision/`), y el **scorecard registra los hiperparámetros de visión**
  (backend, modelo, latencia, válvulas, errores).

> El gold set sigue siendo el **texto original anotado** (es el techo). Lo que se
> evalúa es el costo de leer la nota desde una imagen. Los `doc_id` del manifiesto
> de imágenes deben coincidir con los del gold set, o el harness falla explícito.

## 1. Clonar repositorio

In [ ]:
# Clonar el repo (ajustar la rama si hace falta)
!git clone --branch Agustin --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus

In [ ]:
%cd /content/corpus
%ls

## 2. Instalar dependencias

Además de las dependencias del NER y del harness, se instalan las de la visión
(`M4/vision/requirements.txt`): Pillow, el VLM Qwen2-VL (torch/transformers/
accelerate/qwen-vl-utils) y los backends OCR (Docling y pytesseract).

Para el backend `ocr_tesseract` se necesita además el binario del sistema y el
idioma español (celda opcional).

In [ ]:
!pip install -r M2/run_inference/requirements.txt
!pip install -r M2/harness/requirements.txt
!pip install -r M4/vision/requirements.txt

In [ ]:
# OPCIONAL -- solo si vas a usar el backend ocr_tesseract
# (VLM y Docling no lo necesitan).
# !apt-get install -y tesseract-ocr tesseract-ocr-spa

## 3. Configuración de paths y checks de recursos

Igual que el notebook base, pero se agrega el check del **eval set con imágenes**
de M4. Si no existe (todavía lo construyen Luis/Pau), más abajo hay una celda que
genera un manifiesto de **humo** con una nota sintética para validar el cableado
de punta a punta sin datos reales.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")

checks = {
    "Adaptador LoRA": PROJECT_ROOT / "M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors",
    "Gold set": PROJECT_ROOT / "M2/eval_harness/gold_examples_20.jsonl",
    "Eval set imágenes (M4)": PROJECT_ROOT / "M4/eval_set/casos_imagen.csv",
}
for nombre, path in checks.items():
    print(f"{nombre}: {'OK' if path.exists() else 'FALTA'} -- {path}")

In [ ]:
# .env con PROJECT_ROOT y GROQ_API_KEY (igual que el notebook base).
from google.colab import userdata

with open("/content/corpus/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
    f.write(f"GROQ_API_KEY={userdata.get('GROQ_API_KEY')}\n")
print(".env generado (no versionado, solo vive en este runtime)")

## 4. Elegir el backend de visión

`config_vision.yaml` usa `fuente.backend` para decidir cómo se transcribe la
imagen:

| Backend | Qué es | Requiere |
|---|---|---|
| `vlm` | Qwen2-VL-2B-Instruct | GPU (runtime T4) |
| `ocr_docling` | Docling (baseline OCR) | CPU; descarga modelos la 1ª vez |
| `ocr_tesseract` | Tesseract clásico | binario `tesseract-ocr` + `-spa` |
| `auto` | prueba `fallback_order` | el primero que funcione |

La celda siguiente escribe el backend y el nombre del sistema (que aparece en el
scorecard) directamente en los dos configs, para no tener que editarlos a mano.
Cambiá `BACKEND` y volvé a correr desde aquí para comparar motores.

In [ ]:
import yaml

# --- EDITAR AQUÍ ---
BACKEND = "vlm"  # vlm | ocr_docling | ocr_tesseract | auto
# -------------------

SISTEMA = f"vision_{BACKEND}"
OUTPUT_CACHE = f"M2/predictions/{SISTEMA}.json"

# 1) run_inference/config_vision.yaml -> backend, sistema y output_path
ri_cfg_path = "/content/corpus/M2/run_inference/config_vision.yaml"
ri_cfg = yaml.safe_load(open(ri_cfg_path, encoding="utf-8"))
ri_cfg["sistema"] = SISTEMA
ri_cfg["output_path"] = OUTPUT_CACHE
ri_cfg.setdefault("fuente", {})["tipo"] = "vision"
ri_cfg["fuente"]["backend"] = BACKEND
yaml.safe_dump(ri_cfg, open(ri_cfg_path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

print("run_inference/config_vision.yaml ->")
print(f"  sistema     : {ri_cfg['sistema']}")
print(f"  fuente.tipo : {ri_cfg['fuente']['tipo']}")
print(f"  backend     : {ri_cfg['fuente']['backend']}")
print(f"  output_path : {ri_cfg['output_path']}")
print(f"  manifest    : {ri_cfg['fuente'].get('manifest')}")

### 4.1 (Opcional) Manifiesto de humo con una nota sintética

Corré esta celda **solo si no tenés aún el eval set con imágenes de M4**. Genera
una nota clínica sintética (rotulada, sin datos personales), la guarda en Drive y
escribe un manifiesto de un caso con `doc_id` que existe en el gold set, para
poder correr el flujo completo de prueba. **No es el eval set del equipo.**

In [ ]:
import sys, csv, json
sys.path.insert(0, "/content/corpus/M4/vision")
from demo import crear_nota_sintetica

# doc_id que exista en el gold set (el gold es el techo de la comparación).
gold_path = PROJECT_ROOT / "M2/eval_harness/gold_examples_20.jsonl"
primer_gold = json.loads(open(gold_path, encoding="utf-8").readline())
DOC_ID = primer_gold.get("doc_id", "ex_0")

img_dir = PROJECT_ROOT / "M4/eval_set/imagenes"
img_dir.mkdir(parents=True, exist_ok=True)
img_path = crear_nota_sintetica(img_dir / f"{DOC_ID}.png")

manifest_path = PROJECT_ROOT / "M4/eval_set/casos_imagen.csv"
with open(manifest_path, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "imagen"])
    w.writerow([DOC_ID, str(img_path)])

print(f"Nota sintética: {img_path}")
print(f"Manifiesto de humo: {manifest_path} (doc_id={DOC_ID})")
print("ADVERTENCIA: humo de 1 caso, NO es el eval set real del equipo.")

## 5. Correr inferencia desde imágenes (imagen → OCR/VLM → NER)

Escribe el cache de entidades `M2/predictions/vision_<backend>.json` y, al lado,
el sidecar `vision_<backend>.vision_meta.json` con backend, latencia, válvulas y
errores.

In [ ]:
%cd /content/corpus/M2/run_inference/

In [ ]:
!python run_inference.py --config config_vision.yaml

In [ ]:
# Inspeccionar el sidecar de metadatos de visión
import json
meta_path = PROJECT_ROOT / f"M2/predictions/{SISTEMA}.vision_meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(json.dumps({k: meta[k] for k in (
        "backend", "modelo", "n_casos", "n_ok", "n_error",
        "n_valvula", "latencia_media_s", "latencia_total_s")},
        ensure_ascii=False, indent=2))
else:
    print("No se encontró el sidecar; revisá que fuente.tipo=vision en el config.")

## 6. Correr harness con el config de visión

Usa `M2/harness/config_vision.yaml`, que apunta al cache de visión y escribe los
resultados en `M2/outputs_vision/`. El scorecard (`scorecard_vision.md`) incluye
una sección **Componente visual (M4)** con los hiperparámetros de la corrida.

> Requiere `GROQ_API_KEY` para la Dimensión 3 (LLM-as-judge), igual que el
> baseline. Para saltarla mientras pruebas, usá `--solo exact`.

In [ ]:
%cd /content/corpus/M2/harness

In [ ]:
# Corrida completa (3 dimensiones + scorecard)
!python run_harness.py --config config_vision.yaml

# Alternativa rápida sin Groq (solo exact-match), útil para probar el cableado:
# !python run_harness.py --config config_vision.yaml --solo exact

## 7. Leer el scorecard de visión

Muestra el Markdown generado, con la sección de hiperparámetros de visión arriba
de las métricas globales.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

sc_md = PROJECT_ROOT / "M2/outputs_vision/scorecard_vision.md"
if sc_md.exists():
    display(Markdown(sc_md.read_text(encoding="utf-8")))
else:
    print(f"No existe {sc_md}. ¿Corrió el harness sin --solo?")

## 8. (Opcional) Comparar contra el baseline de texto

El delta de M4 es F1(texto) vs. F1(imagen). Si ya corriste el baseline de texto
(`start_inference_harness.ipynb`), esta celda compara ambos scorecards lado a
lado. El F1 del texto es el techo; la caída al pasar por la imagen, junto con la
latencia y las válvulas, es lo que decide si el componente aporta.

In [ ]:
import json
from pathlib import Path

base = PROJECT_ROOT / "M2/outputs/scorecard_m2.json"
vis = PROJECT_ROOT / "M2/outputs_vision/scorecard_vision.json"

def f1_exact(p):
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8"))["metricas_globales"]["exact_match"]["f1"]

f_base, f_vis = f1_exact(base), f1_exact(vis)
print(f"F1 exact-match texto (techo) : {f_base}")
print(f"F1 exact-match visión        : {f_vis}")
if f_base is not None and f_vis is not None:
    print(f"Delta (visión - texto)       : {f_vis - f_base:+.4f}")

if vis.exists():
    v = json.loads(vis.read_text(encoding="utf-8")).get("vision")
    if v:
        print("\nCosto de operar la visión:")
        print(f"  backend={v['backend']} | latencia media={v['latencia_media_s']}s/caso")
        print(f"  válvulas={v['n_valvula']} | errores={v['n_error']} de {v['n_casos']} casos")